# Primeros pasos

## Máster Universitario en *Big Data* y Computación en la Nube

### Desarrollo y Despliegue de Soluciones *Big Data*

#### Profesorado

* Juan Carlos Alfaro Jiménez

En esta libreta, se explican los conceptos básicos de `mlflow` para gestionar y organizar experimentos en soluciones de *big data*, centrándose en la creación, consulta y gestión de experimentos.

---

## Requisitos previos

Lectura del archivo `README.md` donde se explica:

* Creación de una cuenta de usuario en `Databricks`
* Importación del repositorio al *workspace* de `Databricks`
* Instalación de dependencias en la libreta

---

## Librerías

En primer lugar, se importarán las librerías necesarias para ejecutar esta libreta:

In [0]:
# Standard
import os
from pathlib import Path

# Third party
import mlflow

---

## 1. Introducción

`mlflow` es una plataforma de código abierto que permite gestionar de forma unificada todas las etapas de un proyecto de aprendizaje automático: desde la **experimentación inicial** hasta el **despliegue** y la **monitorización en producción**.

Está diseñada para garantizar la **trazabilidad**, la **reproducibilidad** y la **gobernanza** de los modelos, facilitando el trabajo coordinado entre equipos de ciencia de datos e ingeniería.

Además, se trata de una herramienta versátil: puede ejecutarse en entornos locales, clústeres propios o nubes públicas, y se integra con sistemas de almacenamiento, orquestadores de *pipelines* (`prefect`, `airflow`) y librerías de aprendizaje automático (`scikit-learn`, `xgboost`, `tensorflow`).

##### Componentes principales

`mlflow` se organiza en cuatro módulos clave:

* `Tracking`: registro y visualización de experimentos, con parámetros, métricas, artefactos y modelos asociados.
* `Projects`: definición de entornos portables para reproducir experimentos de manera consistente.
* `Models`: formato estándar para empaquetar y desplegar modelos en diferentes entornos de ejecución.
* `Registry`: sistema centralizado para versionar y gestionar modelos con estados, historial y metadatos.


##### Beneficios

Gracias a esta estructura, los equipos pueden:

* Organizar y comparar experimentos de forma sistemática.
* Reproducir entrenamientos en distintos entornos.
* Versionar y gestionar modelos de manera centralizada.
* Desplegar modelos en producción de forma consistente.
* Almacenar y documentar artefactos vinculados al ciclo de vida.
* Colaborar de manera más eficiente y auditable.

## 2. Servidor de seguimiento

En `mlflow`, el **servidor de seguimiento** es el componente encargado de almacenar y centralizar toda la información de los experimentos: parámetros, métricas, artefactos y modelos.

Cuando trabajamos en local, este servidor puede configurarse de dos formas (véase la [documentación oficial](https://mlflow.org/docs/latest/ml/tracking/server/$0)):

* **Local (por defecto)**: si no se especifica nada, `mlflow` usará el directorio de trabajo actual y crea en él las carpetas necesarias para guardar los experimentos.

* **Remoto o dedicado**: si se proporciona la dirección de un servidor de `mlflow` en ejecución, de manera que los resultados se almacenan de forma centralizada y accesible desde diferentes entornos.

En `Databricks`, en cambio, no es necesario iniciar un servidor manualmente, ya que la plataforma incluye un **servidor de seguimiento integrado**.

### 2.1. Conectando la `API` al servidor de seguimiento

Para registrar y consultar experimentos, `mlflow` necesita saber **a qué dirección debe conectarse**. Esto se define mediante el *tracking* `URI`, que se puede configurar de distintas formas:

* **Definirlo en la variable de entorno `MLFLOW_TRACKING_URI`**, lo que funciona como un ajuste global: la dirección queda guardada en el entorno de ejecución y cualquier *script*, libreta o proceso de `mlflow` que se ejecute en esa sesión utilizará automáticamente esa configuración.  

* **Configurar el *tracking* `URI` directamente en `mlflow`** con la función `mlflow.set_tracking_uri`, una opción muy práctica en libretas: aplica la dirección indicada a todas las operaciones de `mlflow` dentro de la sesión.  

* **Pasar el *tracking* `URI` al crear el cliente**, la forma más explícita: indica directamente al objeto `MlflowClient` el servidor al que debe conectarse mediante el parámetro `tracking_uri`.  

En `Databricks`, este paso no es necesario, ya que la plataforma establece automáticamente la variable de entorno `MLFLOW_TRACKING_URI` apuntando al servidor integrado:

In [0]:
os.environ["MLFLOW_TRACKING_URI"]

De esta manera, cuando interactuemos con `mlflow`, todo quedará almacenado en el servidor de seguimiento correspondiente y podremos consultar los resultados en la interfaz adecuada:

* **En local o en un servidor dedicado**: los resultados se registran en la dirección del *tracking* `URI` que hayamos configurado, y se pueden explorar a través de la interfaz *web* de `mlflow`.

* **En `Databricks`**: no es necesario configurar nada, ya que la plataforma establece automáticamente el *tracking* `URI` y los resultados aparecen directamente en la pestaña `Experiments` de la barra lateral.

---

## 3. Experimentos

Una vez conectado el servidor de seguimiento, la siguiente pieza clave en `mlflow` son los **experimentos**. Un experimento es la unidad lógica que agrupa ejecuciones relacionadas. Cada ejecución corresponde a un entrenamiento o prueba de un modelo, junto con los hiperparámetros, métricas y artefactos que se registran durante el proceso.

Los experimentos permiten **organizar proyectos e iteraciones**, evitando que todo quede mezclado en un mismo espacio. Por ejemplo, podemos tener un experimento dedicado a un modelo de **detección de fraude** y otro distinto para un **sistema de recomendación de películas**. Así, cada conjunto de ejecuciones queda bien separado y es mucho más sencillo comparar resultados dentro de un mismo proyecto.

Hacerlo de esta forma aporta varias ventajas:

* **Mejor organización**: permite mantener separadas las pruebas de distintos proyectos o escenarios, y facilita comparar ejecuciones de manera coherente, algo especialmente útil en proyectos de gran escala.

* **Metadatos enriquecidos**: cada experimento pueden incorporar etiquetas que ayudan a documentar, clasificar y asociar las ejecuciones con un proyecto, equipo o contexto específico.

Como regla general:

* Todas las ejecuciones que utilizan el **mismo conjunto de datos de entrada** deberían agruparse en el **mismo experimento**.
* Para capturar **otras dimensiones de clasificación**, es más adecuado usar **etiquetas**.

Supongamos que una entidad financiera quiere entrenar modelos para **detectar transacciones fraudulentas**:

##### Experimentos

* Un experimento para transacciones realizadas con **tarjeta en línea**.
* Otro experimento para transacciones en **cajeros automáticos**.

Cada experimento se centra en un conjunto de datos distinto y mantiene sus ejecuciones separadas.

##### Ejecuciones

Dentro del experimento de pagos en línea podemos lanzar varias ejecuciones, por ejemplo:

* una con un modelo de regresión logística,
* otra con un árbol de decisión,
* y otra con una red neuronal.

Cada ejecución guarda sus parámetros, métricas y artefactos para que los resultados sean comparables.

##### Etiquetas

Las etiquetas sirven para añadir metadatos generales que faciliten organizar, documentar y filtrar tanto experimentos como ejecuciones. Algunos ejemplos son:

* nombre del proyecto,
* canal de transacción,
* equipo responsable.

Estas etiquetas no cambian los resultados del entrenamiento, pero aportan contexto y hacen mucho más sencillo recuperar y comparar información en proyectos grandes.

### 3.1. Creando experimentos

En `mlflow`, crear un experimento es tan sencillo como asignarle un **nombre** y, opcionalmente, un conjunto de **etiquetas** que aporten contexto:

In [0]:
# Unique experiment name, recommended to be descriptive
experiment = "Online Transactions Fraud Detection"

# This path construction is necessary because the experiment name must be a valid file system path
path = Path(".").absolute().parent.parent / experiment
name = str(path)

# Metadata values to be used as tags
channel = "Online"  # Transaction channel being analyzed
team = "Data Science"  # Responsible team for this experiment

# Optional tags used as metadata
# This information helps to classify and facilitate later search
tags = {"channel": channel, "team": team}

# Create the experiment
# This function returns a unique experiment identifier
identifier = mlflow.create_experiment(name = name, tags = tags)

print(f"Experimento '{name}' creado con identificador: {identifier}")

### 3.2 Consultando y gestionando experimentos

Una vez creado un experimento, `mlflow` ofrece varias operaciones para **consultar su información** y **gestionar su estado** sin necesidad de lanzar ejecuciones. Esto es útil para mantener organizado el espacio de trabajo y documentar los proyectos de manera clara.

#### Consultar un experimento

Podemos recuperar la información básica de un experimento a partir de su identificador:


In [0]:
# Retrieve the experiment using its identifier
# This function returns an object containing all the information about the experiment
experiment = mlflow.get_experiment(identifier)

# Extract relevant information from the experiment
name = experiment.name  # Name of the experiment
identifier = experiment.experiment_id  # Unique identifier of the experiment
location = experiment.artifact_location  # Location where the artifacts associated with the experiment are stored
stage = experiment.lifecycle_stage  # Current stage of the experiment
tags = experiment.tags  # Tags associated with the experiment

print(f"Experimento '{name}' recuperado con identificador: {identifier}")
print(f"Ubicacion de los artefactos: {location}")
print(f"Etapa del experimento: {stage}")
print(f"Etiquetas del experimento: {tags}")

#### Listar todos los experimentos

También es posible ver todos los experimentos registrados en el servidor de seguimiento:

In [0]:
# Retrieve all experiments in the tracking server
# This function returns a list of objects containing metadata about each experiment
experiments = mlflow.search_experiments()

for experiment in experiments:
    # Extract the experiment's unique name and identifier
    name = experiment.name
    identifier = experiment.experiment_id

    print(f"Experimento '{name}' recuperado con identificador: {identifier}")

#### Cambiar el estado de un experimento

Los experimentos en `mlflow` pueden tener dos estados: `active` o `deleted`. Si se elimina un experimento:

In [0]:
# Delete the experiment using its identifier
mlflow.delete_experiment(identifier)

Su estado cambiará a `deleted`, lo que indica que ya no está disponible para su visualización. Después de eliminarlo, se puede comprobar el estado del experimento:

In [0]:
experiment = mlflow.get_experiment(identifier)
stage = experiment.lifecycle_stage

print(stage)

---

## 4. Conclusiones

La [`API` de `mlflow`](https://mlflow.org/docs/3.2.0/api_reference/python_api/index.html) ofrece muchas más opciones para la **gestión y creación de experimentos**, pero en esta libreta solo se han explorado las nociones básicas. Estas son suficientes para entender cómo crear y gestionar experimentos, lo cual es fundamental para comenzar un proyecto.
